In [118]:
import secrets
from dotenv import load_dotenv, find_dotenv
from doc_chat.rag.document_loader import DocumentLoader
from doc_chat.rag.vector_store import VectorStore
from doc_chat.llm import create_llm

from langchain.chains import StuffDocumentsChain, LLMChain
from langchain_core.prompts import PromptTemplate

_ = load_dotenv(find_dotenv())
llm = create_llm()

In [2]:
# load and split document
file_path = "../data/pdf/The Hundred-Page Machine Learning Book.pdf"
doc_loader = DocumentLoader(file_path)
documents, splits = doc_loader.load_and_split()
print("Number of documents: " + str(len(documents)))
print("Number of splits: " + str(len(splits)))

Number of documents: 152
Number of splits: 385


In [3]:
# instantiate vector store
token = secrets.token_urlsafe(16)
vector_store = VectorStore(documents=splits, token=token)

In [154]:
# get documents
question = "Explain logistic regression"
documents = vector_store.retrieve_documents(question, k=2)
print("Number of documents: " + str(len(documents)))
for i, doc in enumerate(documents):
    print(f"Document {i}")
    print(f"Page: {doc.metadata['page']}")
    print(f"Content: {doc.page_content[:200]}")
    print("-----")

Number of documents: 2
Document 0
Page: 32
Content: 3.
By looking at the graph of the standard logistic function, we can see how well it ﬁts our
classiﬁcation purpose: if we optimize the values ofx and b appropriately, we could interpret
the output off
-----
Document 1
Page: 32
Content: Figure 3: Standard logistic function.
At the time where the absence of computers required scientists to perform manual calculations,
they were eager to ﬁnd a linear classiﬁcation model. They ﬁgured ou
-----


In [165]:
prompt = PromptTemplate.from_template(
"""
You are an assistant who answers questions **only** from the texts provided below.

Citation rules
1. After the answer, append the ID of each text you used, wrapped in square brackets.
   • Example (single source):  The river is the Seine. [TXT2]
   • Example (multiple sources):  Coffee was first cultivated in Yemen. [TXT3][TXT4]
2. If the answer cannot be found in the texts, reply exactly:  I don’t know

{context}

"Question:```{question}```"
"""
)

chain = prompt | llm

In [177]:
context = "[TXT1] Paris is the capital of France.\n [TXT2] It was founded in the 3rd century BC."
query = "When was France’s capital founded?"
result = chain.invoke({"context": context, "question": query })
result

'\nAnswer: Paris was founded in the 3rd century BC. [TXT2]'

In [178]:
# format documents
context_parts = []
for i, doc in enumerate(documents):
    # context_parts.append(f"page: {doc.metadata['page']}\n")
    context_parts.append(f"[TXT{i+1}]\n")
    context_parts.append(f"{doc.page_content}\n\n")

context = ''.join(context_parts)

query = "Explain logistic regression"                                       # :contentReference[oaicite:3]{index=3}
result = chain.invoke({"context": context, "question": query })
result

'\nLogistic regression is a type of classification model that is used to predict the probability of a certain outcome based on input variables. It works by optimizing the values of x and b to interpret the output of f(x) as the probability of a positive outcome. The model uses a threshold of 0.5 to determine whether the outcome is positive or negative, but this threshold can vary depending on the problem. The standard logistic function, also known as the sigmoid function, is commonly used in logistic regression because it has a codomain of (0,1) and can accurately classify data points as positive or negative. [TXT1][TXT2]'